# Mapping Surface Urban Heat Island Change in Greater Concepción, Chile (2015-2026)

This notebook implements a reproducible **Google Earth Engine + Python** workflow to map changes in **Surface Urban Heat Island (SUHI)** intensity in Greater Concepción, Chile, between summer 2015 and summer 2026.

The study area includes the municipalities of **Concepción, Talcahuano, Hualpén, and San Pedro de la Paz**. The analysis uses Landsat 8 Collection 2 Level 2 imagery, QA_PIXEL cloud masking, land surface temperature (LST), NDVI, MNDWI, NDBI, a spectral-index urban mask, and a spectral-index rural reference mask.

SUHI is calculated as each urban LST pixel minus the mean rural reference LST for the same summer period. The result is a relative land-surface thermal signal, not an air-temperature measurement.


## 1. Install dependencies

This cell detects whether the notebook is running in Google Colab. In Colab, it installs missing packages. In a local Python environment, it does not install or reinstall packages.


In [ ]:
import importlib
import subprocess
import sys


def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


IN_COLAB = running_in_colab()

PACKAGE_IMPORTS = {
    "earthengine-api": "ee",
    "geemap": "geemap",
    "geopandas": "geopandas",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "rasterio": "rasterio",
    "shapely": "shapely",
    "contextily": "contextily",
    "folium": "folium",
    "branca": "branca",
}

if IN_COLAB:
    missing_packages = []
    for package_name, import_name in PACKAGE_IMPORTS.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            missing_packages.append(package_name)

    if missing_packages:
        print("Installing missing packages:", ", ".join(missing_packages))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
    else:
        print("All required packages are already available in Colab.")
else:
    print("Local environment detected. Install requirements.txt before running if packages are missing.")


## 2. Imports and output folders

This section imports the required packages and creates the output folders used by the workflow.


In [ ]:
from pathlib import Path
import warnings
import base64
import io

import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import rasterio.warp
import rasterio.windows
import contextily as cx
import folium
import branca.colormap as bcm
from IPython.display import display
from matplotlib.colors import TwoSlopeNorm

warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
GEOTIFF_DIR = OUTPUT_DIR / "geotiffs"
TABLE_DIR = OUTPUT_DIR / "tables"
DOCS_DIR = Path("docs")

for folder in [FIGURE_DIR, GEOTIFF_DIR, TABLE_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Output folders ready.")


## 3. Initialize Google Earth Engine

Authenticate and initialize Earth Engine. The notebook supports both Google Colab and local Jupyter environments.


In [ ]:
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("Earth Engine initialized successfully.")


## 4. Load study area

The study area boundary is stored as a local shapefile at `data/admin_boundaries/GreaterConcepcion.shp`. The file is derived from IDE Chile DPA 2023 and represents Concepción, Talcahuano, Hualpén, and San Pedro de la Paz.


In [ ]:
AOI_PATH = Path("data/admin_boundaries/GreaterConcepcion.shp")
EXPECTED_MUNICIPALITIES = ["Concepción", "Talcahuano", "Hualpén", "San Pedro de la Paz"]

if not AOI_PATH.exists():
    raise FileNotFoundError(
        f"AOI shapefile not found: {AOI_PATH}. "
        "Place GreaterConcepcion.shp and its companion files in data/admin_boundaries/."
    )

aoi_gdf = gpd.read_file(AOI_PATH)

if aoi_gdf.crs is None:
    raise ValueError("The AOI shapefile has no CRS. Define the CRS before running this notebook.")

aoi_gdf = aoi_gdf.to_crs("EPSG:4326")

metadata_columns = [col for col in ["COMUNA", "NAME", "PROVINCIA", "REGION"] if col in aoi_gdf.columns]
print("Municipalities represented in the study area:")
for municipality in EXPECTED_MUNICIPALITIES:
    print(f"- {municipality}")

if metadata_columns:
    display(aoi_gdf[metadata_columns].drop_duplicates())
else:
    display(aoi_gdf.drop(columns="geometry", errors="ignore").head())

print(f"Number of AOI features: {len(aoi_gdf)}")
print(f"CRS: {aoi_gdf.crs}")

AOI_FC = geemap.geopandas_to_ee(aoi_gdf)
AOI_GEOM = AOI_FC.geometry()

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(AOI_FC, {"color": "red"}, "Greater Concepción study area")
Map


## 5. Landsat 8 preprocessing functions

The functions below apply QA_PIXEL cloud masking, Landsat Collection 2 Level 2 scale factors, LST conversion to degrees Celsius, spectral index calculation, and summer composite generation.


In [ ]:
LANDSAT_COLLECTION = "LANDSAT/LC08/C02/T1_L2"
MAX_CLOUD_COVER = 60
EXPORT_SCALE = 30

SUMMER_2015_START = "2015-01-01"
SUMMER_2015_END = "2015-03-31"
SUMMER_2026_START = "2026-01-01"
SUMMER_2026_END = "2026-03-31"

URBAN_NDVI_MAX = 0.45
URBAN_NDBI_MIN = 0.02
RURAL_NDVI_MIN = 0.60
RURAL_NDBI_MAX = -0.10

FIGURE_EXTENT = {
    "xmin": -73.18,
    "xmax": -72.98,
    "ymin": -36.88,
    "ymax": -36.68,
}
FIGURE_BOUNDS = [FIGURE_EXTENT["xmin"], FIGURE_EXTENT["ymin"], FIGURE_EXTENT["xmax"], FIGURE_EXTENT["ymax"]]


def mask_landsat_qa_pixel(image):
    image = ee.Image(image)
    qa = image.select("QA_PIXEL")
    clear_mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )
    return image.updateMask(clear_mask).copyProperties(image, ["system:time_start"])


def apply_landsat_scale_factors(image):
    image = ee.Image(image)
    optical = image.select("SR_B.").multiply(0.0000275).add(-0.2)
    thermal = image.select("ST_B10").multiply(0.00341802).add(149.0).rename("ST_B10")
    return image.addBands(optical, None, True).addBands(thermal, None, True)


def add_lst_celsius(image):
    image = ee.Image(image)
    lst = image.select("ST_B10").subtract(273.15).rename("LST")
    return image.addBands(lst, None, True)


def add_ndvi(image):
    image = ee.Image(image)
    ndvi = image.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
    return image.addBands(ndvi, None, True)


def add_mndwi(image):
    image = ee.Image(image)
    mndwi = image.normalizedDifference(["SR_B3", "SR_B6"]).rename("MNDWI")
    return image.addBands(mndwi, None, True)


def add_ndbi(image):
    image = ee.Image(image)
    ndbi = image.normalizedDifference(["SR_B6", "SR_B5"]).rename("NDBI")
    return image.addBands(ndbi, None, True)


def prepare_landsat_image(image):
    image = ee.Image(image)
    image = mask_landsat_qa_pixel(image)
    image = apply_landsat_scale_factors(image)
    image = add_lst_celsius(image)
    image = add_ndvi(image)
    image = add_mndwi(image)
    image = add_ndbi(image)
    return image


def build_summer_composite(start_date, end_date, label):
    collection = (
        ee.ImageCollection(LANDSAT_COLLECTION)
        .filterBounds(AOI_GEOM)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("PROCESSING_LEVEL", "L2SP"))
        .filter(ee.Filter.lte("CLOUD_COVER", MAX_CLOUD_COVER))
        .map(prepare_landsat_image)
    )
    image_count = collection.size().getInfo()
    print(f"{label}: {image_count} Landsat 8 scenes")
    if image_count == 0:
        raise ValueError(f"No Landsat 8 scenes found for {label}.")
    return collection.median().clip(AOI_GEOM)


print("Landsat preprocessing functions ready. Earth Engine image casting is enabled.")


## 6. Summer 2015 composite

Build the 2015 summer Landsat 8 composite, print the number of images used, and display an RGB preview.


In [ ]:
image2015 = build_summer_composite(SUMMER_2015_START, SUMMER_2015_END, "Summer 2015")

rgb_vis = {
    "bands": ["SR_B4", "SR_B3", "SR_B2"],
    "min": 0.02,
    "max": 0.30,
    "gamma": 1.2,
}

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(image2015, rgb_vis, "RGB 2015")
Map.addLayer(AOI_FC, {"color": "red"}, "Study area", False)
Map


## 7. Summer 2026 composite

Build the 2026 summer Landsat 8 composite, print the number of images used, and display an RGB preview.


In [ ]:
image2026 = build_summer_composite(SUMMER_2026_START, SUMMER_2026_END, "Summer 2026")

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(image2026, rgb_vis, "RGB 2026")
Map.addLayer(AOI_FC, {"color": "red"}, "Study area", False)
Map


## 8. Spectral indices

Confirm that both composites contain LST, NDVI, MNDWI, and NDBI without duplicated index-band names.


In [ ]:
REQUIRED_ANALYSIS_BANDS = ["LST", "NDVI", "MNDWI", "NDBI"]


def validate_analysis_bands(image, label):
    band_names = image.bandNames().getInfo()
    missing = [band for band in REQUIRED_ANALYSIS_BANDS if band not in band_names]
    duplicated_index_names = [band for band in band_names if band in ["NDVI_1", "MNDWI_1", "NDBI_1", "LST_1"]]
    if missing:
        raise ValueError(f"{label} is missing required bands: {missing}")
    if duplicated_index_names:
        raise ValueError(f"{label} contains duplicated index bands: {duplicated_index_names}")
    print(f"{label} required bands present: {', '.join(REQUIRED_ANALYSIS_BANDS)}")


validate_analysis_bands(image2015, "2015 composite")
validate_analysis_bands(image2026, "2026 composite")

index_vis = {
    "min": -0.5,
    "max": 0.8,
    "palette": ["#7f3b08", "#f6e8c3", "#c7eae5", "#01665e"],
}

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(image2015.select("NDVI"), index_vis, "NDVI 2015", False)
Map.addLayer(image2026.select("NDVI"), index_vis, "NDVI 2026", False)
Map.addLayer(image2015.select("NDBI"), {"min": -0.4, "max": 0.4, "palette": ["#2c7bb6", "#ffffbf", "#d7191c"]}, "NDBI 2015", False)
Map.addLayer(image2026.select("NDBI"), {"min": -0.4, "max": 0.4, "palette": ["#2c7bb6", "#ffffbf", "#d7191c"]}, "NDBI 2026", False)
Map.addLayer(AOI_FC, {"color": "black"}, "Study area", False)
Map


## 9. Urban mask

The urban mask follows the existing spectral-index approach. It excludes water using MNDWI, dense vegetation using NDVI, and non-built-up surfaces using NDBI. A common urban mask is used so 2015 and 2026 are compared over the same built-up-like pixels.


In [ ]:
def build_urban_mask(image):
    return (
        image.select("MNDWI").lt(0.0)
        .And(image.select("NDVI").lt(URBAN_NDVI_MAX))
        .And(image.select("NDBI").gt(URBAN_NDBI_MIN))
    )


urban_mask2015 = build_urban_mask(image2015)
urban_mask2026 = build_urban_mask(image2026)
common_urban_mask = urban_mask2015.And(urban_mask2026).rename("Urban_Mask")

urban_lst2015 = image2015.select("LST").updateMask(common_urban_mask).rename("Urban_LST_2015")
urban_lst2026 = image2026.select("LST").updateMask(common_urban_mask).rename("Urban_LST_2026")
delta_lst = urban_lst2026.subtract(urban_lst2015).rename("Delta_LST")

lst_vis = {
    "min": 18,
    "max": 42,
    "palette": ["#313695", "#4575b4", "#74add1", "#abd9e9", "#ffffbf", "#fdae61", "#f46d43", "#d73027", "#a50026"],
}

delta_lst_vis = {
    "min": -6,
    "max": 6,
    "palette": ["#2166ac", "#67a9cf", "#d1e5f0", "#f7f7f7", "#fddbc7", "#ef8a62", "#b2182b"],
}

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(common_urban_mask.selfMask(), {"palette": ["#fdae61"]}, "Common urban mask", True)
Map.addLayer(urban_lst2015, lst_vis, "Urban LST 2015", False)
Map.addLayer(urban_lst2026, lst_vis, "Urban LST 2026", False)
Map.addLayer(delta_lst, delta_lst_vis, "Delta urban LST 2026 - 2015", False)
Map.addLayer(AOI_FC, {"color": "black"}, "Study area", False)
Map


## 10. Rural reference

The rural reference mask uses vegetated, non-water, non-built-up pixels inside the study area. The mean rural LST is calculated independently for 2015 and 2026.


In [ ]:
def build_rural_mask(image):
    return (
        image.select("NDVI").gt(RURAL_NDVI_MIN)
        .And(image.select("NDBI").lt(RURAL_NDBI_MAX))
        .And(image.select("MNDWI").lt(0.0))
    )


def reduce_mean(image, band_name, geometry=AOI_GEOM):
    result = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geometry,
        scale=EXPORT_SCALE,
        maxPixels=1e13,
        bestEffort=True,
    )
    return ee.Number(result.get(band_name))


rural_mask2015 = build_rural_mask(image2015)
rural_mask2026 = build_rural_mask(image2026)

rural_lst2015 = image2015.select("LST").updateMask(rural_mask2015).rename("Rural_LST_2015")
rural_lst2026 = image2026.select("LST").updateMask(rural_mask2026).rename("Rural_LST_2026")

rural_mean2015 = reduce_mean(rural_lst2015, "Rural_LST_2015")
rural_mean2026 = reduce_mean(rural_lst2026, "Rural_LST_2026")

print("Mean rural LST 2015 (deg C):", rural_mean2015.getInfo())
print("Mean rural LST 2026 (deg C):", rural_mean2026.getInfo())

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(rural_mask2015.selfMask(), {"palette": ["#1a9850"]}, "Rural reference pixels 2015", False)
Map.addLayer(rural_mask2026.selfMask(), {"palette": ["#006837"]}, "Rural reference pixels 2026", True)
Map.addLayer(AOI_FC, {"color": "black"}, "Study area", False)
Map


## 11. Surface Urban Heat Island intensity

SUHI is calculated as:

```text
SUHI_2015 = Urban_LST_2015 - Mean_Rural_LST_2015
SUHI_2026 = Urban_LST_2026 - Mean_Rural_LST_2026
```


In [ ]:
suhi2015 = urban_lst2015.subtract(rural_mean2015).rename("SUHI_2015")
suhi2026 = urban_lst2026.subtract(rural_mean2026).rename("SUHI_2026")

suhi_vis = {
    "min": -2,
    "max": 8,
    "palette": ["#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026"],
}

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(suhi2015, suhi_vis, "SUHI 2015", True)
Map.addLayer(suhi2026, suhi_vis, "SUHI 2026", False)
Map.addLayer(AOI_FC, {"color": "black"}, "Study area", False)
Map


## 12. Delta SUHI

Delta SUHI is calculated as:

```text
Delta_SUHI = SUHI_2026 - SUHI_2015
```

Positive values mean increased relative urban heat island intensity. Negative values mean decreased relative urban heat island intensity. This is land surface temperature, not air temperature.


In [ ]:
delta_suhi = suhi2026.subtract(suhi2015).rename("Delta_SUHI")

delta_suhi_vis = {
    "min": -4,
    "max": 4,
    "palette": ["#2166ac", "#67a9cf", "#d1e5f0", "#f7f7f7", "#fddbc7", "#ef8a62", "#b2182b"],
}

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(delta_suhi, delta_suhi_vis, "Delta SUHI 2026 - 2015", True)
Map.addLayer(AOI_FC, {"color": "black"}, "Study area", False)
Map


## 13. Summary statistics

Calculate mean, minimum, maximum, and standard deviation for urban LST, rural reference LST, SUHI, and change layers. The table is exported to `outputs/tables/summary_statistics.csv`.


In [ ]:
def summary_stats(image, band_name, label):
    reducer = (
        ee.Reducer.minMax()
        .combine(ee.Reducer.mean(), sharedInputs=True)
        .combine(ee.Reducer.stdDev(), sharedInputs=True)
    )
    result = image.reduceRegion(
        reducer=reducer,
        geometry=AOI_GEOM,
        scale=EXPORT_SCALE,
        maxPixels=1e13,
        bestEffort=True,
    ).getInfo()
    return {
        "layer": label,
        "mean": result.get(f"{band_name}_mean"),
        "min": result.get(f"{band_name}_min"),
        "max": result.get(f"{band_name}_max"),
        "standard_deviation": result.get(f"{band_name}_stdDev"),
    }


stats = [
    summary_stats(urban_lst2015, "Urban_LST_2015", "Urban LST 2015"),
    summary_stats(urban_lst2026, "Urban_LST_2026", "Urban LST 2026"),
    summary_stats(delta_lst, "Delta_LST", "Delta LST"),
    summary_stats(rural_lst2015, "Rural_LST_2015", "Mean rural LST 2015"),
    summary_stats(rural_lst2026, "Rural_LST_2026", "Mean rural LST 2026"),
    summary_stats(suhi2015, "SUHI_2015", "SUHI 2015"),
    summary_stats(suhi2026, "SUHI_2026", "SUHI 2026"),
    summary_stats(delta_suhi, "Delta_SUHI", "Delta SUHI"),
]

summary_df = pd.DataFrame(stats)
summary_path = TABLE_DIR / "summary_statistics.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")
summary_df


## 14. Export GeoTIFFs

Export the main raster layers as local GeoTIFFs at 30 m resolution, clipped to the study area. If a GeoTIFF download fails, the notebook reports the error and continues so the publication figures can still be generated directly from the Earth Engine images.


In [ ]:
def export_geotiff(image, filename, region=AOI_GEOM):
    output_path = GEOTIFF_DIR / filename
    geemap.ee_export_image(
        image,
        filename=str(output_path),
        scale=EXPORT_SCALE,
        region=region,
        file_per_band=False,
    )
    print(f"Saved: {output_path}")


figure_geom = ee.Geometry.Rectangle(FIGURE_BOUNDS, proj="EPSG:4326", geodesic=False)

geotiff_exports = {
    "LST_2015.tif": urban_lst2015,
    "LST_2026.tif": urban_lst2026,
    "Delta_LST.tif": delta_lst,
    "SUHI_2015.tif": suhi2015,
    "SUHI_2026.tif": suhi2026,
    "Delta_SUHI.tif": delta_suhi,
}

try:
    export_geotiff(
        image2015.select(["SR_B4", "SR_B3", "SR_B2"]).rename(["Red", "Green", "Blue"]),
        "RGB_2015.tif",
        region=figure_geom,
    )
except Exception as export_error:
    print(f"GeoTIFF export skipped for RGB_2015.tif: {export_error}")

for filename, image in geotiff_exports.items():
    try:
        export_geotiff(image, filename)
    except Exception as export_error:
        print(f"GeoTIFF export skipped for {filename}: {export_error}")


## 15. Generate publication-quality figures

Create publication-quality PNG figures from the exported GeoTIFFs. The analysis AOI remains unchanged; the fixed display extent below is used only for cartographic presentation and keeps the urban area visually prominent.


In [ ]:
FIGURE_DPI = 300
STUDY_AOI = aoi_gdf.copy()
WEB_MERCATOR = "EPSG:3857"
BASEMAP_SOURCE = cx.providers.CartoDB.Positron

municipality_labels = [
    {"name": "Talcahuano", "lon": -73.115, "lat": -36.725},
    {"name": "Hualpen", "lon": -73.095, "lat": -36.785},
    {"name": "Concepcion", "lon": -73.050, "lat": -36.825},
    {"name": "San Pedro de la Paz", "lon": -73.105, "lat": -36.845},
]

REQUIRED_FIGURES = [
    "Figure_01_Study_Area.png",
    "Figure_02_RGB_2015.png",
    "Figure_03_LST_2015.png",
    "Figure_04_LST_2026.png",
    "Figure_05_Delta_LST.png",
    "Figure_06_SUHI_2015.png",
    "Figure_07_SUHI_2026.png",
    "Figure_08_Delta_SUHI.png",
    "Figure_09_Comparison_LST_SUHI.png",
]


def project_lonlat_extent(extent_dict):
    transformer = rasterio.warp.transform_bounds(
        "EPSG:4326",
        WEB_MERCATOR,
        extent_dict["xmin"],
        extent_dict["ymin"],
        extent_dict["xmax"],
        extent_dict["ymax"],
        densify_pts=21,
    )
    return transformer


def add_source_note(fig, y=0.02):
    fig.text(
        0.02,
        y,
        "Source: Landsat 8 Collection 2 Level 2 / Google Earth Engine; IDE Chile DPA 2023. LST is land surface temperature, not air temperature.",
        fontsize=8,
        ha="left",
    )


def add_north_arrow(ax):
    ax.annotate(
        "N",
        xy=(0.94, 0.90),
        xytext=(0.94, 0.79),
        xycoords="axes fraction",
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", color="#222222", lw=1.4),
    )


def add_scale_bar(ax, length_km=5):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    bar_length = length_km * 1000
    x_start = x0 + 0.08 * (x1 - x0)
    y_start = y0 + 0.08 * (y1 - y0)
    ax.plot([x_start, x_start + bar_length], [y_start, y_start], color="#222222", lw=3, solid_capstyle="butt")
    ax.text(x_start + bar_length / 2, y_start + 0.018 * (y1 - y0), f"{length_km} km", ha="center", va="bottom", fontsize=8, color="#222222")


def add_labels(ax):
    labels_gdf = gpd.GeoDataFrame(
        municipality_labels,
        geometry=gpd.points_from_xy([item["lon"] for item in municipality_labels], [item["lat"] for item in municipality_labels]),
        crs="EPSG:4326",
    ).to_crs(WEB_MERCATOR)
    for _, row in labels_gdf.iterrows():
        ax.text(
            row.geometry.x,
            row.geometry.y,
            row["name"],
            fontsize=8,
            color="#222222",
            ha="center",
            va="center",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.72),
        )


def style_cartographic_axis(ax, title):
    ax.set_title(title, fontsize=15, fontweight="bold", pad=12)
    ax.set_axis_off()
    ax.set_aspect("equal")
    xmin, ymin, xmax, ymax = project_lonlat_extent(FIGURE_EXTENT)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    add_north_arrow(ax)
    add_scale_bar(ax)


def finite_percentile_limits(array, low=2, high=98):
    values = array[np.isfinite(array)]
    if values.size == 0:
        raise ValueError("The image array contains no finite values.")
    return np.percentile(values, [low, high])


def symmetric_limits(array, percentile=98):
    values = array[np.isfinite(array)]
    if values.size == 0:
        raise ValueError("The image array contains no finite values.")
    limit = np.percentile(np.abs(values), percentile)
    if limit == 0:
        limit = 1
    return -limit, limit


def add_context(ax):
    cx.add_basemap(ax, source=BASEMAP_SOURCE, crs=WEB_MERCATOR, attribution=False, zoom=12)
    STUDY_AOI.to_crs(WEB_MERCATOR).boundary.plot(ax=ax, color="#111111", linewidth=1.0, alpha=0.9, zorder=5)
    add_labels(ax)


def read_raster_for_display(filename, band=1):
    path = GEOTIFF_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Required raster not found: {path}")
    with rasterio.open(path) as src:
        window = rasterio.windows.from_bounds(*FIGURE_BOUNDS, transform=src.transform)
        data = src.read(band, window=window, boundless=True, fill_value=np.nan).astype("float32")
        transform = src.window_transform(window)
        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = np.nan
        data[data < -9990] = np.nan
        bounds = rasterio.windows.bounds(window, src.transform)
        projected_bounds = rasterio.warp.transform_bounds(src.crs, WEB_MERCATOR, *bounds, densify_pts=21)
    return data, projected_bounds


def read_rgb_for_display(filename):
    path = GEOTIFF_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Required RGB raster not found: {path}. Run section 14 to export RGB_2015.tif.")
    with rasterio.open(path) as src:
        window = rasterio.windows.from_bounds(*FIGURE_BOUNDS, transform=src.transform)
        rgb = src.read([1, 2, 3], window=window, boundless=True, fill_value=np.nan).astype("float32")
        bounds = rasterio.windows.bounds(window, src.transform)
        projected_bounds = rasterio.warp.transform_bounds(src.crs, WEB_MERCATOR, *bounds, densify_pts=21)
    rgb = np.moveaxis(rgb, 0, -1)
    rgb = np.clip((rgb - 0.02) / (0.30 - 0.02), 0, 1)
    rgb[~np.isfinite(rgb)] = np.nan
    return rgb, projected_bounds


def save_study_area_figure():
    fig, ax = plt.subplots(figsize=(8, 8))
    add_context(ax)
    style_cartographic_axis(ax, "Study Area and Urban Display Extent")
    add_source_note(fig)
    fig.text(0.02, 0.00, "Demonstration workflow; not an operational climate-risk product.", fontsize=8, ha="left")
    plt.tight_layout(rect=[0, 0.06, 1, 1])
    output_path = FIGURE_DIR / "Figure_01_Study_Area.png"
    plt.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close()
    print(f"Saved: {output_path}")


def save_rgb_figure():
    rgb, bounds = read_rgb_for_display("RGB_2015.tif")
    fig, ax = plt.subplots(figsize=(9, 8))
    add_context(ax)
    ax.imshow(rgb, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], origin="upper", alpha=0.88, zorder=3)
    style_cartographic_axis(ax, "Landsat 8 RGB Composite - Summer 2015")
    add_source_note(fig)
    fig.text(0.02, 0.00, "Display extent is cartographic only; analysis uses the full study AOI.", fontsize=8, ha="left")
    plt.tight_layout(rect=[0, 0.06, 1, 1])
    output_path = FIGURE_DIR / "Figure_02_RGB_2015.png"
    plt.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close()
    print(f"Saved: {output_path}")


def save_raster_figure(input_file, output_file, title, cmap, colorbar_label, diverging=False, fixed_limits=None, alpha=0.78):
    array, bounds = read_raster_for_display(input_file)
    if fixed_limits is not None:
        vmin, vmax = fixed_limits
    elif diverging:
        vmin, vmax = symmetric_limits(array)
    else:
        vmin, vmax = finite_percentile_limits(array)

    fig, ax = plt.subplots(figsize=(9, 8))
    add_context(ax)

    if diverging:
        norm = TwoSlopeNorm(vcenter=0, vmin=vmin, vmax=vmax)
        image = ax.imshow(array, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], origin="upper", cmap=cmap, norm=norm, alpha=alpha, zorder=3)
    else:
        image = ax.imshow(array, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], origin="upper", cmap=cmap, vmin=vmin, vmax=vmax, alpha=alpha, zorder=3)

    style_cartographic_axis(ax, title)

    colorbar = plt.colorbar(image, ax=ax, shrink=0.75)
    colorbar.set_label(colorbar_label)

    add_source_note(fig)
    fig.text(0.02, 0.00, "Demonstration workflow; not an operational climate-risk product.", fontsize=8, ha="left")
    plt.tight_layout(rect=[0, 0.06, 1, 1])
    output_path = FIGURE_DIR / output_file
    plt.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close()
    print(f"Saved: {output_path}")


def save_comparison_figure():
    titles = ["Urban LST 2015", "Urban LST 2026", "Delta SUHI 2026 - 2015"]
    cmaps = ["inferno", "inferno", "RdBu_r"]
    labels = ["LST (deg C)", "LST (deg C)", "Delta SUHI (deg C)"]
    rasters = [
        read_raster_for_display("LST_2015.tif"),
        read_raster_for_display("LST_2026.tif"),
        read_raster_for_display("Delta_SUHI.tif"),
    ]

    lst_values = np.concatenate([rasters[0][0].ravel(), rasters[1][0].ravel()])
    lst_values = lst_values[np.isfinite(lst_values)]
    lst_vmin, lst_vmax = np.percentile(lst_values, [2, 98])
    delta_vmin, delta_vmax = symmetric_limits(rasters[2][0])

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    for index, ax in enumerate(axes):
        array, bounds = rasters[index]
        add_context(ax)
        if index < 2:
            image = ax.imshow(array, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], origin="upper", cmap=cmaps[index], vmin=lst_vmin, vmax=lst_vmax, alpha=0.78, zorder=3)
        else:
            norm = TwoSlopeNorm(vcenter=0, vmin=delta_vmin, vmax=delta_vmax)
            image = ax.imshow(array, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], origin="upper", cmap=cmaps[index], norm=norm, alpha=0.78, zorder=3)
        style_cartographic_axis(ax, titles[index])
        colorbar = plt.colorbar(image, ax=ax, shrink=0.7)
        colorbar.set_label(labels[index])

    fig.suptitle("Surface Urban Heat Island Change in Greater Concepción, Chile", fontsize=17, fontweight="bold")
    add_source_note(fig)
    fig.text(0.02, 0.00, "Demonstration workflow; not an operational climate-risk product.", fontsize=8, ha="left")
    plt.tight_layout(rect=[0, 0.06, 1, 0.94])
    output_path = FIGURE_DIR / "Figure_09_Comparison_LST_SUHI.png"
    plt.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close()
    print(f"Saved: {output_path}")


save_study_area_figure()
save_rgb_figure()
save_raster_figure("LST_2015.tif", "Figure_03_LST_2015.png", "Urban Land Surface Temperature - 2015", "inferno", "LST (deg C)", fixed_limits=(18, 42))
save_raster_figure("LST_2026.tif", "Figure_04_LST_2026.png", "Urban Land Surface Temperature - 2026", "inferno", "LST (deg C)", fixed_limits=(18, 42))
save_raster_figure("Delta_LST.tif", "Figure_05_Delta_LST.png", "Urban LST Change: 2026 - 2015", "RdBu_r", "Delta LST (deg C)", diverging=True)
save_raster_figure("SUHI_2015.tif", "Figure_06_SUHI_2015.png", "Surface Urban Heat Island Intensity - 2015", "RdYlBu_r", "SUHI (deg C)", fixed_limits=(-2, 8))
save_raster_figure("SUHI_2026.tif", "Figure_07_SUHI_2026.png", "Surface Urban Heat Island Intensity - 2026", "RdYlBu_r", "SUHI (deg C)", fixed_limits=(-2, 8))
save_raster_figure("Delta_SUHI.tif", "Figure_08_Delta_SUHI.png", "SUHI Change: 2026 - 2015", "RdBu_r", "Delta SUHI (deg C)", diverging=True)
save_comparison_figure()

missing_figures = [filename for filename in REQUIRED_FIGURES if not (FIGURE_DIR / filename).exists()]
if missing_figures:
    raise FileNotFoundError(f"Missing expected figure files: {missing_figures}")

print("All publication figures were generated successfully:")
for filename in REQUIRED_FIGURES:
    print(f"- {FIGURE_DIR / filename}")


## 16. Interactive web map

Create a standalone interactive HTML map for GitHub Pages. The map is centered on the urban display extent and saved as `docs/index.html`.


In [ ]:
def array_to_data_url(array, cmap_name, vmin=None, vmax=None, diverging=False, alpha=0.72):
    values = array.copy()
    mask = ~np.isfinite(values)
    if vmin is None or vmax is None:
        if diverging:
            vmin, vmax = symmetric_limits(values)
        else:
            vmin, vmax = finite_percentile_limits(values)

    if diverging:
        norm = TwoSlopeNorm(vcenter=0, vmin=vmin, vmax=vmax)
    else:
        norm = plt.Normalize(vmin=vmin, vmax=vmax)

    cmap = plt.get_cmap(cmap_name)
    rgba = cmap(norm(values))
    rgba[..., 3] = np.where(mask, 0, alpha)
    rgba = (rgba * 255).astype("uint8")

    buffer = io.BytesIO()
    plt.imsave(buffer, rgba, format="png")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"


def add_raster_overlay(map_object, filename, layer_name, cmap_name, colorbar_caption, show=False, fixed_limits=None, diverging=False):
    array, _ = read_raster_for_display(filename)
    if fixed_limits is not None:
        vmin, vmax = fixed_limits
    elif diverging:
        vmin, vmax = symmetric_limits(array)
    else:
        vmin, vmax = finite_percentile_limits(array)

    data_url = array_to_data_url(array, cmap_name, vmin=vmin, vmax=vmax, diverging=diverging)
    folium.raster_layers.ImageOverlay(
        image=data_url,
        bounds=[[FIGURE_EXTENT["ymin"], FIGURE_EXTENT["xmin"]], [FIGURE_EXTENT["ymax"], FIGURE_EXTENT["xmax"]]],
        name=layer_name,
        opacity=0.72,
        interactive=True,
        cross_origin=False,
        zindex=3,
        show=show,
    ).add_to(map_object)

    colors = [plt.get_cmap(cmap_name)(i / 10) for i in range(11)]
    hex_colors = ["#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255)) for r, g, b, _ in colors]
    colormap = bcm.LinearColormap(hex_colors, vmin=vmin, vmax=vmax, caption=colorbar_caption)
    colormap.add_to(map_object)


map_center = [
    (FIGURE_EXTENT["ymin"] + FIGURE_EXTENT["ymax"]) / 2,
    (FIGURE_EXTENT["xmin"] + FIGURE_EXTENT["xmax"]) / 2,
]

interactive_map = folium.Map(
    location=map_center,
    zoom_start=12,
    tiles="CartoDB positron",
    control_scale=True,
)

folium.GeoJson(
    STUDY_AOI.to_crs("EPSG:4326").to_json(),
    name="Study area boundary",
    style_function=lambda feature: {"color": "#111111", "weight": 2, "fillOpacity": 0},
).add_to(interactive_map)

add_raster_overlay(interactive_map, "LST_2015.tif", "LST 2015", "inferno", "LST 2015 (deg C)", show=False, fixed_limits=(18, 42))
add_raster_overlay(interactive_map, "LST_2026.tif", "LST 2026", "inferno", "LST 2026 (deg C)", show=False, fixed_limits=(18, 42))
add_raster_overlay(interactive_map, "Delta_LST.tif", "Delta LST", "RdBu_r", "Delta LST (deg C)", show=False, diverging=True)
add_raster_overlay(interactive_map, "SUHI_2015.tif", "SUHI 2015", "RdYlBu_r", "SUHI 2015 (deg C)", show=False, fixed_limits=(-2, 8))
add_raster_overlay(interactive_map, "SUHI_2026.tif", "SUHI 2026", "RdYlBu_r", "SUHI 2026 (deg C)", show=False, fixed_limits=(-2, 8))
add_raster_overlay(interactive_map, "Delta_SUHI.tif", "Delta SUHI 2026 minus 2015", "RdBu_r", "Delta SUHI (deg C)", show=True, diverging=True)

folium.LayerControl(collapsed=False).add_to(interactive_map)

map_note = (
    '<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999; '
    'background: white; padding: 10px; border: 1px solid #999; font-size: 12px; max-width: 320px;">'
    '<b>Surface Urban Heat Island Change</b><br>'
    'Landsat LST is land surface temperature, not air temperature. '
    'Display extent is cartographic only; analysis uses the full study AOI.'
    '</div>'
)
interactive_map.get_root().html.add_child(folium.Element(map_note))

html_path = DOCS_DIR / "index.html"
interactive_map.save(html_path)
print(f"Saved: {html_path}")
interactive_map


## 17. Interpretation and limitations

The final **Delta SUHI** layer shows where relative urban heat island intensity increased or decreased between summer 2015 and summer 2026.

- Positive Delta SUHI values indicate increased relative urban heat island intensity.
- Negative Delta SUHI values indicate decreased relative urban heat island intensity.
- LST is land surface temperature, not air temperature.
- SUHI is relative to the rural reference defined in this workflow.
- Results depend on cloud masking, rural reference definition, spectral thresholds, Landsat 30 m resolution, coastal influence, and season selection.
- This is a demonstration workflow, not an operational climate-risk product.
